# 능동학습 실습

**Active Learning · AL**

다음에 정답을 확보할 데이터를 모델이 선택하고 새 결과로 학습을 갱신하는 방법.

소재 분야에서 이해하기: 예측이 불확실한 소재를 계산한 뒤 모델에 추가 학습시킨다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [베이지안 능동학습 연구](https://www.nature.com/articles/s41467-020-19597-w)

## 1. 다음에 무엇을 측정할지 모델이 고릅니다

무작위로 고르는 것과 불확실성이 큰 곳을 고르는 것을 같은 예산으로 비교합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel

def truth(x):
    return np.sin(4 * np.pi * x) * np.exp(-2 * x)

pool = np.linspace(0, 1, 400)[:, None]
target = truth(pool[:, 0])

def make_model():
    return GaussianProcessRegressor(kernel=ConstantKernel(1.0) * RBF(0.08),
                                    normalize_y=True, alpha=1e-4, random_state=0)

print('후보 %d개 중 20번의 측정 예산으로 전체 곡선을 맞춰야 합니다.' % len(pool))

In [ ]:
def run(strategy, budget=20, seed=0):
    local = np.random.default_rng(seed)
    chosen = list(local.choice(len(pool), 3, replace=False))
    errors = []
    for step in range(budget):
        model = make_model().fit(pool[chosen], target[chosen])
        mean, std = model.predict(pool, return_std=True)
        errors.append(np.mean(np.abs(mean - target)))
        if strategy == 'random':
            candidate = int(local.choice([i for i in range(len(pool)) if i not in chosen]))
        else:
            masked = std.copy(); masked[chosen] = -1
            candidate = int(np.argmax(masked))
        chosen.append(candidate)
    model = make_model().fit(pool[chosen], target[chosen])
    errors.append(np.mean(np.abs(model.predict(pool) - target)))
    return np.array(errors), chosen

random_errors, random_chosen = run('random')
active_errors, active_chosen = run('uncertainty')
plt.plot(random_errors, 'o-', label='random sampling')
plt.plot(active_errors, 's-', label='active learning')
plt.xlabel('measurements added'); plt.ylabel('mean absolute error'); plt.legend(); plt.show()
print('최종 오차: 무작위 %.4f / 능동학습 %.4f' % (random_errors[-1], active_errors[-1]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
for axis, (name, chosen) in zip(axes, [('random', random_chosen), ('active learning', active_chosen)]):
    model = make_model().fit(pool[chosen], target[chosen])
    mean, std = model.predict(pool, return_std=True)
    axis.plot(pool[:, 0], target, 'k--', label='truth')
    axis.plot(pool[:, 0], mean, label='model')
    axis.fill_between(pool[:, 0], mean - 2 * std, mean + 2 * std, alpha=0.2)
    axis.scatter(pool[chosen, 0], target[chosen], c='red', s=18, zorder=5)
    axis.set_title(name); axis.set_xlabel('x')
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()
print('능동학습은 빠르게 변하는 구간에 측정을 몰아줍니다.')
print('단 잡음이 큰 데이터에서는 불확실성만 보고 고르면 잡음이 큰 지점만 반복 측정할 수 있습니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#active-learning)을 여세요.